# Parameter-efficient Fine-tuning (PEFT): Language Models for Sequence Classification
This notebook explores the parameter-efficient fine-tuning (PEFT) process to fine-tune language models for sequence classification tasks, like sentiment analysis. We focus on textual data. We need to have the following libraries installed:

* `numpy`: needed for handly functions, like computing the _argmax_.
* `torch`: needed to handle GPU-related stuff.
* `bitsandbytes`: needed if you want to use weight quantization on your model.
* `peft`: needed to implement PEFT methods to train models.
* `scikit-learn`: needed to compute metrics to evaluate the model.
* `transformers`: needed to load the base model and tokenizer, and to handle all training-related classes.
* `datasets`: needed to load the datasets from HuggingFace, also useful for efficient training.
* `matplotlib`: needed to plot the confusion matrix at the end of training. You can comment this out if you don't need to do it.

The `transformers`, `datasets`, and `peft` libraries are developed by HuggingFace 🤗, and therefore well integrated, saving us the need of cumbersome preprocessing.


### What is Parameter-efficient Fine-tuning (PEFT)?
Parameter-efficient fine-tuning (PEFT) refers to a set of methods aimed to fine-tune language models (and neural networks, in general) focusing on a smaller set of parameters compared to full fine-tuning, _freezing_ all other parameters. The main goal is to make the process more efficient, i.e., less time-consuming and lighter on resources.

There are several PEFT techniques, here we implement:

* **Low-Rank Adaptation (LoRA)** [(Hu, Edward J., et al, 2021)](https://arxiv.org/abs/2106.09685), one of the most widely used PEFT techniques. It is based on the principle of low-rank matrix decomposition. Instead of updating the full weight matrices of the model, LoRA adds trainable low-rank matrices to the existing weights. This approach allows the model to adapt to new tasks by learning a low-dimensional update, which is then added to the fixed, pre-trained weights.

* **Quantized LoRA (QLoRA)** [(Dettmers, Tim, et al., 2023)](https://arxiv.org/abs/2305.14314), conceptually very similar to LoRA, but in addition it quantizes the weights of the model to usually 4 bits. It reduces precision, but it makes training much lighter on the GPU.

For very well done introductions to PEFT, you can take a look at these articles [[1]](https://www.ibm.com/think/topics/parameter-efficient-fine-tuning#:~:text=Parameter-efficient%20fine-tuning%20%28PEFT%29%20is%20a%20method%20of%20improving,neural%20networks%20for%20specific%20tasks%20or%20data%20sets.), [[2]](https://huggingface.co/docs/peft/index).

> Note that when you fine-tune a model using LoRA/QLoRA, you are _not_ tuning the parameters of the base model, but only the weights of the adapters. Therefore, instead of exporting the entire base model with the fine-tuned weights, you will only export the (much lighter) LoRA adapters. These can be injected back into the base model to obtain the fine-tuned model.

In [12]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
    TaskType
)
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## Configs
Here, we can set a few configuration parameters, like the model's path, and LoRA and training-related hyperparameters.

Regarding **model choice**, basically any language model can be used for sequence classification, as long as it is loaded using the `transformers.AutoModelForSequenceClassification` class. Even generative Large Language Models (LLMs) can be used as classifiers, as the generative head gets swapped for a classification head, that outputs a vector of the same dimension of the number of unique classes.

> #### Target Modules
> There are some important considerations to make for the `TARGET_MODULES` parameter. It should be a list containing the names of the modules you want to target with LoRA adapters. However, note that each model gives its own names to its layers, there is no one-fits-all solution for this. It is common practice to target the $W_{Query}$, $W_{Key}$, $W_{Values}$, and sometimes $W_{Out}$ weight matrices of the Transformer, even if this is not guaranteed to yield the best results. To see how the model names these matrices, just print your model.

In [ ]:
MODEL_ID         : str   = "FacebookAI/xlm-roberta-base" # path to model's Hugging Face repository
DATASET_ID       : str   = "istat-ai/sentipolc_dataset"  # path to dataset's Hugging Face repository

QUANTIZATION     : bool  = False                         # whether to 4bit-quantize the model or not
LORA_RANK        : int   = 8                             # LoRA rank
LORA_ALPHA       : int   = 32                            # LoRA scaling parameter
LORA_DROPOUT     : float = 0.1                           # dropout rate for LoRA layers
TARGET_MODULES   : list  = [                             # modules to target for LoRA
    'q_proj', 'k_proj', 'v_proj', 'o_proj'
]

OUTPUT_DIR       : str   = f'saved_models/your_ft_model' # output directory for the saved model
NUM_EPOCHS       : int   = 2                             # number of training epochs
LEARNING_RATE    : float = 2e-4                          # learning rate for weight updates
LR_SCHEDULER     : str   = 'linear'                      # learning rate decay scheduler
OPTIMIZER        : str   = "adamw_torch_fused"           # optimizer to use for training
TRAIN_BATCH_SIZE : int   = 16                            # batch size during training (affects training)
EVAL_BATCH_SIZE  : int   = 16                            # batch size during evaluation (does not affect training)
GA_STEPS         : int   = 2                             # gradient accumulation steps, simulates higher batch size
WARMUP_RATIO     : float = 0.1                           # % of steps during which the lr increases before reaching the specified value
WEIGHT_DECAY     : float = 0.01                          # penalty for large weight values, prevents overfitting
LOGGING_STEPS    : int   = 20                            # how often training metrics are logged (steps)
EVAL_STEPS       : int   = 20                            # how often the model is evaluated (steps)
EVAL_STRATEGY    : str   = 'steps'                       # based on what the model is evaluated
SAVE_STRATEGY    : str   = 'steps'                       # based on what the model is saved
FP16             : bool  = True                          # mixed-precision training (use with older hardware, like T4 GPUs)
BF16             : bool  = False                         # mixed-precision training (use with hardware that supports Ampere+)
LOAD_BEST        : bool  = True                          # load the best performing model at end based on specified metric
REPORT_TO        : list  = []                            # which logging integrations to use (ex. tensorboard)
LOG_LEVEL        : str   = 'warning'                     # controls logging verbosity

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

<hr>

## Dataset

Let's load the dataset from HuggingFace. Your data needs to be a `datasets.DatasetDict` with the following structure to work with this notebook:

```
DatasetDict({
    train: Dataset({
        features: ['text', 'labels']
    })
    eval: Dataset({
        features: ['text', 'labels']
    })
    test: Dataset({
        features: ['text', 'labels']
    })
})
```
A few things:
* The `test` split is optional, it will be used only at the end if you want to cross-validate your model on unseen data.
* The `eval` split is also optional, but we _highly_ recommend providing it to prevent overfitting.
* The `labels` must be integrers. If they are another type, like strings, make sure to preprocess them and encode them into integers.
* If you have any additional features, it's ok, they will just be discarded automatically by the `Trainer`.

We will also extract the number of unique labels from the `train` split.

In [ ]:
data = load_dataset(DATASET_ID)
n_labels = len(data["train"].unique("labels"))

### Tokenization
Now we need to tokenize the data, i.e., encode the text into integers representing token IDs. To do this, we will use the pre-trained tokenizer associated to the model we want to fine-tune. Some models, like Llama LLMs, lack padding token IDs in the tokenizers, therefore we need to manually assign them (we will assign it to the EOS token ID, i.e., the end-of-sentence token). We need a padding token ID to train the model using batches of data, where each batch contains sequences of the same length.

First, let's load the tokenizer and extract the maximum sequence length that the model can process. Then, we can define a tokenize function and apply it to the text feature. The tokenize function will return the following features, in addition to the existing ones:

* `input_ids`: the tokenized sequences.
* `attention_mask`: the tensor used to mask the padding tokens.

These features, in addition to the `labels` feature (**important**: it needs to be exactly named "labels"!), are all we need to train the model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
max_len = tokenizer.model_max_length

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

def tokenize(example, max_len: int):
    return tokenizer(example["text"], padding=True, truncation=True, max_length=max_len, return_tensors="pt")

tokenized_data = data.map(lambda example: tokenize(example, max_len), batched=True)

Even if the `transformers.Trainer` automatically discards irrelevant features in the dataset, it is best practice to clean up the dataset before training. Therefore, we will remove each feature that is not in `['input_ids', 'attention_mask', 'labels']`.

Moreover, we need to make sure that every column is cast correctly as a PyTorch Tensor.

In [11]:
relevant_features = ['input_ids', 'attention_mask', 'labels']

for split in tokenized_data.column_names:
    for feature in tokenized_data[split].column_names:
        if feature not in relevant_features:
            tokenized_data[split] = tokenized_data[split].remove_columns(feature)

tokenized_data.set_format(type='torch', columns=relevant_features)

<hr>

### Model Preparation
Compared to standard (full) fine-tuning, with parameter-efficient fine-tuning (PEFT) we need to do some preprocessing on our model to prepare it for training. We can choose to _quantize_ the model's weights to 4 bit.